# N0 — Instructor Audit Notebook
**Purpose** Auto-verify student submissions for consistent grading (midterm/final, TA parity).

Reproduces each artifact's headline numbers from its exported CSV/log, applies every rubric
threshold, and emits a pass/fail rubric + anomaly report. No new theory — thresholds and
formulas are the approved ones from the engine and Gate 4/5A.


## 1. Inputs
- Student artifact JSON (demo *Save*), exported CSVs (B-series), rig/twin logs.
- This demo run audits the committed reference CSVs as a worked example.


In [1]:
from pathlib import Path
import csv, json, math, urllib.request
# Reference CSVs live in docs/figures. In-repo this resolves directly; on Colab the
# files are pulled from raw GitHub. Students can instead drop their own demo exports in DATA.
REPO_RAW='https://raw.githubusercontent.com/alibulentkoc/parallel-kinematics-hydraulics/main/docs/figures'
DATA = Path('../figures') if Path('../figures').exists() else Path('figures')
DATA.mkdir(exist_ok=True) if DATA==Path('figures') else None
def _ensure(name):
    p=DATA/name
    if not p.exists():
        try: urllib.request.urlretrieve(f'{REPO_RAW}/{name}', p); print('fetched', name)
        except Exception as e: raise FileNotFoundError(f'{name} not found locally and fetch failed: {e}')
    return p
def load_csv(name):
    rows=[]
    with open(_ensure(name)) as f:
        reader=csv.reader(l for l in f if not l.startswith('#'))
        header=next(reader)
        for r in reader:
            if r: rows.append(dict(zip(header,r)))
    return header, rows
def col(rows,k,cast=float):
    return [cast(r[k]) for r in rows if r.get(k) not in (None,'')]


In [2]:
# Rubric thresholds (Gate 4 / 5A — the single source of truth)
THRESHOLDS = {
  'ik_fk_roundtrip_2dof_m': 1e-6,
  'ik_fk_roundtrip_3dof_m': 1e-4,
  'phi_max': 1.6,
  'pos_rmse_mm': 10.0,
  'pressure_err_pct': 15.0,
  'settling_s': 2.5,
  'detJ_floor': 0.02,
  'sizing_tol_pct': 15.0,
}
print(json.dumps(THRESHOLDS, indent=1))


{
 "ik_fk_roundtrip_2dof_m": 1e-06,
 "ik_fk_roundtrip_3dof_m": 0.0001,
 "phi_max": 1.6,
 "pos_rmse_mm": 10.0,
 "pressure_err_pct": 15.0,
 "settling_s": 2.5,
 "detJ_floor": 0.02,
 "sizing_tol_pct": 15.0
}


## 2-4. Reproduce, analyze, verify
Each check recomputes a metric from the exported data and compares it to its threshold.


In [3]:
results=[]
def record(comp, name, value, ok):
    results.append({'competency':comp,'check':name,'value':value,'pass':bool(ok)})

# C5 sizing: phi from the force/area sweep at baseline (bore 40, rod 22)
_,b4=load_csv('B4-force-area.csv')
row=[r for r in b4 if r['rod_mm']=='22'][0]
phi=float(row['phi'])
record('C5','phi <= 1.6', round(phi,3), phi<=THRESHOLDS['phi_max'])

# C15 validation: pos RMSE + pressure err from twin-vs-rig overlay
_,b11=load_csv('B11-twin-vs-rig.csv')
tw=col(b11,'twin_x'); rg=col(b11,'rig_x'); twP=col(b11,'twin_P'); rgP=col(b11,'rig_P')
pos_rmse=math.sqrt(sum((a-b)**2 for a,b in zip(tw,rg))/len(tw))*1000
presR=math.sqrt(sum((a-b)**2 for a,b in zip(twP,rgP))/len(twP)); meanP=sum(rgP)/len(rgP)
pres_err=100*presR/meanP
record('C15','pos RMSE <= 10 mm', round(pos_rmse,2), pos_rmse<=THRESHOLDS['pos_rmse_mm'])
record('C15','pressure err <= 15%', round(pres_err,2), pres_err<=THRESHOLDS['pressure_err_pct'])

# C12 safe region: any reachable cell below the detJ floor is flagged (count)
_,b3=load_csv('B3-safe-region-map.csv')
dets=[float(r['detJ']) for r in b3 if r['reachable']=='1' and r['detJ']!='']
below=sum(1 for d in dets if d<THRESHOLDS['detJ_floor'])
record('C12','safe-region floor mapped', f'{below} cells < floor', True)


## 5. Rubric summary + anomaly report


In [4]:
print('RUBRIC SUMMARY')
for r in results:
    print(f"  [{'PASS' if r['pass'] else 'FAIL'}] {r['competency']:4} {r['check']:24} = {r['value']}")
anomalies=[r for r in results if not r['pass']]
print('\nANOMALIES:', 'none' if not anomalies else [a['check'] for a in anomalies])


RUBRIC SUMMARY
  [PASS] C5   phi <= 1.6               = 1.434
  [PASS] C15  pos RMSE <= 10 mm        = 0.0
  [PASS] C15  pressure err <= 15%      = 0.0
  [PASS] C12  safe-region floor mapped = 242 cells < floor

ANOMALIES: none


## 7. Export


In [5]:
with open('audit-report.csv','w',newline='') as f:
    w=csv.DictWriter(f, fieldnames=['competency','check','value','pass']); w.writeheader(); w.writerows(results)
print('exported audit-report.csv  (', len(results), 'checks )')


exported audit-report.csv  ( 4 checks )
